# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to explore the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) defined by a Croissant schema.

<br/>
### Dataset Source
The dataset metadata is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed. > Colab: uncomment if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

Inspect the metadata for high-level dataset information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display metadata overview
print("---- DATASET METADATA ----")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished if hasattr(dataset.metadata, 'datePublished') else 'N/A'}")
print(f"Authors: {dataset.metadata.author if hasattr(dataset.metadata, 'author') else 'N/A'}")
print(f"Version: {dataset.metadata.version if hasattr(dataset.metadata, 'version') else 'N/A'}")

## 2. Data Overview
Review available *record sets*, their fields, and discoverability using their `@id`. This helps you target the desired record set/fields for your exploration.

A **record set** in Croissant is a collection of records (analogous to a table), and each `field`/`column` is defined with a unique `@id`.

In [ ]:
# List all record sets in the dataset, showing their @id and available fields
print("---- RECORD SETS ----")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"RecordSet @id: {record_set.metadata['@id']}")
    print(f"  Name: {record_set.metadata.get('name', 'N/A')}")
    print(f"  Description: {record_set.metadata.get('description', 'N/A')}")
    print(f"  Fields (@id):")
    for field in record_set.fields:
        print(f"    - {field['@id']} (name={field.get('name', '')} type={field.get('dataType', '')})")
    print("-")

## 3. Data Extraction
Let's extract one or more record sets by their `@id`. Below, we'll load selected record sets into pandas DataFrames for further manipulation.

**Choose record set(s) to extract based on the overview above.**

In [ ]:
# Example: Extract all record sets into DataFrames by @id
#
dataframes = {}
record_set_ids = [rs.metadata['@id'] for rs in record_sets]
for rs in record_sets:
    rs_id = rs.metadata['@id']
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Example rows:\n{df.head()}\n")
# For demonstration, print the first (if only one) record set's DataFrame columns
default_rs_id = record_set_ids[0] if record_set_ids else None
if default_rs_id:
    print(f"Primary columns in {default_rs_id}: {dataframes[default_rs_id].columns.tolist()}")
    display(dataframes[default_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering on a numeric field, normalizing values, and optionally grouping by a chosen attribute. All columns/fields must be referenced by their `@id`.

Change the variables below to point to relevant target fields by their `@id` from the DataFrame extracted above.

In [ ]:
# Example EDA: Using the first record set as the main table

# Select the primary record set
main_record_set_id = default_rs_id
df = dataframes[main_record_set_id].copy() if main_record_set_id else None

# Set your numeric field @id for analysis (update if necessary based on printed columns above)
numeric_field_id = None
if df is not None:
    # Try to auto-pick a field
    for c in df.columns:
        # Try to guess a numeric column (float or int type, or named coverage/abundance/mw etc)
        sample = df[c].dropna()
        if (sample.dtype in ['float64', 'int64']) or (any(k in c.lower() for k in ['coverage', 'count', 'mw', 'abundance'])):
            numeric_field_id = c
            break

if df is not None and numeric_field_id is not None:
    print(f"Selected numeric field: {numeric_field_id}")

    # 1. Filter records where numeric_field > threshold
    threshold = df[numeric_field_id].quantile(0.5)  # Median as dynamic threshold (change as needed)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")
    display(filtered_df.head())
    # 2. Normalize numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (Z-score):")
    display(filtered_df[[numeric_field_id, normalized_col]].head())
    # 3. Group by a categorical (string) field by @id
    group_field_id = None
    for col in filtered_df.columns:
        if col != numeric_field_id and filtered_df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("Grouped by mean:")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA. Please select one above.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field. If grouping was possible, display the group means as a barplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we leveraged `mlcroissant` to load, explore, and process a FAIR^2-compliant Croissant dataset describing the mass spectrometry analysis of extracellular vesicle proteins from human mast cells.

- We demonstrated metadata extraction and dynamic table discovery using `@id` references.
- Data records were loaded and fields processed solely by their unique `@id` values, ensuring robust, schema-compliant analytic workflows.
- You can extend this workflow by referencing additional field `@id`s, deeper feature engineering, or integrating with downstream ML pipelines using Croissant datasets!

**Next steps:** Use the schema's field definitions (`@id`, `dataType`) for precise feature selection, and explore related datasets by their Croissant schema for interoperability.